# 🧠 STAR General Alpha — Zero-Shot Neuro-Symbolic Knowledge Graphs
### *by [Reagent Systems](https://github.com/reagent-systems)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThyFriendlyFox/star/blob/main/STAR_General_Alpha_Colab.ipynb)

**No training required.** Point this at ANY text dataset and it automatically:
1. Extracts key concepts from text
2. Discovers relationships using zero-shot classification
3. Builds a symbolic knowledge graph with forward-chaining inference

> ⚙️ **Runtime:** Go to *Runtime → Change runtime type* → Select **A100** + **High RAM**

## 0 · GPU Check

In [ ]:
import torch, os, psutil

assert torch.cuda.is_available(), "No GPU detected — please enable A100 in Runtime settings."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
ram_gb  = psutil.virtual_memory().total / 1e9

print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")
print(f"RAM  : {ram_gb:.1f} GB")

if "A100" in gpu_name:
    print("✅ A100 detected — perfect.")
else:
    print(f"⚠️  Expected A100, got {gpu_name} — will still work but slower.")

if ram_gb > 50:
    print("✅ High RAM enabled.")
else:
    print("⚠️  Consider enabling High RAM for large datasets.")

## 1 · Install Dependencies

In [ ]:
!pip install -q torch numpy datasets transformers tqdm networkx matplotlib pydot

## 2 · Clone STAR Repository

In [ ]:
import os
if not os.path.isdir("star"):
    !git clone https://github.com/ThyFriendlyFox/star.git
%cd star
!mkdir -p out

---
# Demo 1 · SemEval Relation Dataset (small, fast)

First, let’s test on a small well-known dataset to see the pipeline in action.
This uses [SemEval 2010 Task 8](https://huggingface.co/datasets/SemEvalWorkshop/sem_eval_2010_task_8) — relation classification between entity pairs.

In [ ]:
!python prototypes/prototype_general_alpha.py \
    --dataset SemEvalWorkshop/sem_eval_2010_task_8 \
    --text-field sentence \
    --label-field relation \
    --max-samples 50 \
    --device cuda \
    --min-confidence 0.25 \
    --export-json out/kb_semeval.json \
    --export-dot out/kb_semeval.dot \
    --trace-inference

### Visualize the SemEval Knowledge Graph

In [ ]:
!python prototypes/view_kb_graph.py out/kb_semeval.json --out out/semeval_graph.png --show --edge-labels --figscale 2.0
from IPython.display import Image, display
display(Image("out/semeval_graph.png"))

---
# Demo 2 · Competitive Programming Dataset (large, streaming)

Now let’s point it at a completely different domain — [Generative Coding Dataset](https://huggingface.co/datasets/grandsmile/Generative_Coding_Dataset).
~29GB of competitive programming problems. We stream it and sample 100 problems.

The model has **never seen this data** and receives **no training** — it discovers algorithmic relationships purely through zero-shot classification.

In [ ]:
!python prototypes/prototype_general_alpha.py \
    --dataset grandsmile/Generative_Coding_Dataset \
    --text-field question \
    --label-field tags \
    --streaming \
    --max-samples 100 \
    --device cuda \
    --min-confidence 0.25 \
    --relations "implements algorithm,solves problem type,has complexity" \
    --export-json out/kb_coding.json \
    --export-dot out/kb_coding.dot \
    --trace-inference

### Visualize the Coding Knowledge Graph

In [ ]:
!python prototypes/view_kb_graph.py out/kb_coding.json --out out/coding_graph.png --show --edge-labels --figscale 2.5
from IPython.display import Image, display
display(Image("out/coding_graph.png"))

---
# Demo 3 · Try Your Own Dataset

Change the dataset, text field, and labels below to test on anything from HuggingFace:

In [ ]:
# 🔧 Change these to try your own dataset!
DATASET = "ag_news"           # Any HuggingFace dataset
TEXT_FIELD = "text"           # Column containing text
LABEL_FIELD = "label"         # Optional: column with categories
MAX_SAMPLES = 50              # How many documents to process
RELATIONS = ""                # Optional: comma-separated custom relations

cmd = f"""python prototypes/prototype_general_alpha.py \\
    --dataset {DATASET} \\
    --text-field {TEXT_FIELD} \\
    {"--label-field " + LABEL_FIELD if LABEL_FIELD else ""} \\
    --max-samples {MAX_SAMPLES} \\
    --device cuda \\
    --min-confidence 0.25 \\
    {"--relations '" + RELATIONS + "'" if RELATIONS else ""} \\
    --export-json out/kb_custom.json \\
    --export-dot out/kb_custom.dot \\
    --trace-inference"""

!{cmd}

---
## 📝 How It Works

1. **Concept Extraction** — Regex-based noun phrase extraction identifies key entities in each document
2. **Zero-Shot Classification** — `facebook/bart-large-mnli` classifies relationships between entity pairs without any training
3. **Symbolic Graduation** — High-confidence predictions are promoted into a formal knowledge base
4. **Forward Chaining** — Inference engine applies rules (transitivity, abstraction) to derive new knowledge
5. **Vector Store** — Entity embeddings enable semantic similarity queries

**No training. No fine-tuning. Just pre-trained transformers + symbolic reasoning.**

---
*Built by [Reagent Systems](https://github.com/reagent-systems) · [STAR Repository](https://github.com/ThyFriendlyFox/star)*